In [ ]:
%matplotlib widget

import ipywidgets as widgets
from ipywidgets import HBox, VBox, Box, Layout, Output, Label
from IPython.display import display, Latex, Image, Markdown

import params as st
from model import linSys, sysMatrices
from feedforward import feedForwardStep, trajectory, feedForwardFlat
from regulator import acker

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
def make_box_layout():
     return widgets.Layout(
        border='solid 1px black',
        margin='0px 5px 5px 0px',
        padding='2px 2px 2px 2px'
     )

# Steuerung mit und ohne Regelung beim Gleichstrommotor

In [ ]:
imag = Image("../images/dcMotor.png", width=300)

outL = Output()
outR = Output()

with outL:
    display(imag)
with outR:
    display(Markdown("""
**Modellgleichungen**

\\begin{align*}
    L_\\text{A} \\dot{i}(t) & = -R\\, i(t) - k\\, \\omega(t) + u(t) \\\\
    J \\dot{\\omega}(t) & = k\\, i(t) - \\nu\\, \\omega(t) - \\tau_\\text{L}(t)
\\end{align*}

**Flachheitsbasierte Steuerung mit flachem Ausgang $y = \\omega$**

Entwurf |   | Eingang
----------------------|----|------------------------
\\begin{align*}
    \\omega_{\\mathrm{r}}(t) & = \\begin{cases}
    0, & t \\le t_0 \\\\
    \\omega_{\\mathrm{d}} \\varphi_2((t - t_0)/T) & t_0 < t < t_0 + T \\\\
    \\omega_{\\mathrm{d}} & t \\ge t_0 + T
    \\end{cases}
\\end{align*} |   | \\begin{align*}
    i_{\\mathrm{r}}(t) & = \\frac{J \\dot{\\omega}_{\\mathrm{r}}(t) + \\nu \\omega_{\\mathrm{r}}(t)}{k} \\\\
    u_{\\mathrm{r}}(t) & = L_\\text{A} \\dot{i}_{\\mathrm{r}}(t) + R\\, i_{\\mathrm{r}}(t) + k\\, \\omega_{\\mathrm{r}}(t)
\\end{align*}

**Folgeregelung mit Zustandsregler**
\\begin{align*}
    u(t) & = u_{\\mathrm{r}}(t) - \\mathbf{k}^\\intercal (\\mathbf{x}(t) - \\mathbf{x}_{\\mathrm{r}}(t)), &
    \\mathbf{x}_{\\mathrm{r}} & = (i_{\\mathrm{r}}, \\omega_{\\mathrm{r}})^\\intercal
\\end{align*}

Die Steuerung wird mit den nominellen Parametern entworfen, die Strecke kann davon abweichen und durch ein konstantes Lastmoment $\\tau_\\text{L} = d_0$ ab $t = t_\\text{L}$ gestört werden.
Die reine Zustandsrückführung reduziert den Einfluss von Parameterfehlern und Lastmoment, eine bleibende Regelabweichung wird erst durch eine Störgrößenkompensation (siehe Regulator) beseitigt.
    """))
cols = HBox([outL, outR], layout=Layout(display='flex', flex_flow='row', justify_content='space-around', align_items='center'))
display(cols)

**Definition Parameter**

In [ ]:
paramsNom = st.R, st.LA, st.k, st.J, st.nu
A, b, c, _, _ = sysMatrices(paramsNom)
tSim = np.linspace(0, 6, 1201)
tL = 4

In [ ]:
def control(t, x, mode, K, t0, T, wd):
    if mode == 'Sprung':
        return feedForwardStep(t0, wd)(t)
    xRef, uRef = feedForwardFlat(t, wd, t0, T)
    if mode == 'Steuerung':
        return uRef
    return uRef - K @ (x - xRef)

In [ ]:
output = widgets.Output()

with output:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.subplots_adjust(bottom=0.08, top=0.93, left=0.1, right=0.9, hspace=0.1)

ax1.set_xlim([tSim[0], tSim[-1]])
for ax in [ax1, ax2, ax3]:
    ax.grid()
ax1.set_ylabel(r"$u$ in V")
ax2.set_ylabel(r"$\omega$ in rad/s")
ax3.set_ylabel(r"$i$ in A")
ax3.set_xlabel(r"$t$ in s")

lineU, = ax1.plot([], [], 'C0')
lineURef, = ax1.plot([], [], 'C1--')
lineW, = ax2.plot([], [], 'C0', label=r"Istverlauf")
lineWRef, = ax2.plot([], [], 'C1--', label=r"Sollverlauf")
lineI, = ax3.plot([], [], 'C0')
lineIRef, = ax3.plot([], [], 'C1--')

handlesAx, labelsAx = ax2.get_legend_handles_labels()
fig.legend(handlesAx, labelsAx,
           bbox_to_anchor=(0.1, 0.94, 0.8, .15), loc=3,
           ncol=2, mode="expand", borderaxespad=0., framealpha=0.5)

sliderT0 = widgets.FloatSlider(value=0.5,
                               min=0,
                               max=2,
                               step=0.5,
                               description=r'$t_0$')
sliderT = widgets.FloatSlider(value=1,
                              min=0.2,
                              max=3,
                              step=0.1,
                              description=r'$T$')
sliderWd = widgets.FloatSlider(value=10,
                               min=1,
                               max=30,
                               step=1,
                               description=r"$\omega_\text{d}$")
sliderDJ = widgets.FloatSlider(value=0,
                               min=-50,
                               max=50,
                               step=5,
                               description=r"$\Delta J$ in %")
sliderDR = widgets.FloatSlider(value=0,
                               min=-50,
                               max=50,
                               step=5,
                               description=r"$\Delta R$ in %")
sliderD0 = widgets.FloatSlider(value=0,
                               min=0,
                               max=0.5,
                               step=0.05,
                               description=r"$d_0$")
sliderPoleK = widgets.FloatSlider(value=-20,
                                  min=-50,
                                  max=-2,
                                  step=1,
                                  description=r"$s_{\text{R}}$")
radioU = widgets.RadioButtons(options=['Sprung', 'Steuerung', 'Steuerung mit Regelung'],
                              value='Steuerung',
                              disabled=False)

def calcSystem(_):
    t0 = sliderT0.value
    T = sliderT.value
    wd = sliderWd.value
    d0 = sliderD0.value
    mode = radioU.value

    paramsSys = (st.R * (1 + sliderDR.value / 100), st.LA, st.k,
                 st.J * (1 + sliderDJ.value / 100), st.nu)
    K = acker(A, b, [sliderPoleK.value] * 2)
    tauL = lambda t: d0 if t >= tL else 0

    def sys(t, x):
        u = lambda _t: control(_t, x, mode, K, t0, T, wd)
        return linSys(t, x, u, tauL, paramsSys)

    res = solve_ivp(sys,
                    [tSim[0], tSim[-1]],
                    [0, 0],
                    t_eval=tSim,
                    max_step=1e-2)

    uAll = np.array([control(t, res.y[:, idx], mode, K, t0, T, wd) for idx, t in enumerate(res.t)])
    ref = [feedForwardFlat(t, wd, t0, T) for t in res.t]
    xRef = np.array([r[0] for r in ref])
    uRef = np.array([r[1] for r in ref])

    lineU.set_data(res.t, uAll)
    lineW.set_data(res.t, res.y[1])
    lineI.set_data(res.t, res.y[0])
    if mode == 'Sprung':
        lineURef.set_data([], [])
        lineIRef.set_data([], [])
        lineWRef.set_data(res.t, [wd if t >= t0 else 0 for t in res.t])
    else:
        lineURef.set_data(res.t, uRef)
        lineIRef.set_data(res.t, xRef[:, 0])
        lineWRef.set_data(res.t, xRef[:, 1])

    for ax, data in [(ax1, np.concatenate([uAll, uRef])),
                     (ax2, np.concatenate([res.y[1], xRef[:, 1]])),
                     (ax3, np.concatenate([res.y[0], xRef[:, 0]]))]:
        dMin, dMax = np.min(data), np.max(data)
        ax.set_ylim(dMin - 0.1 * np.abs(dMax - dMin) - 1e-3, dMax + 0.1 * np.abs(dMax - dMin) + 1e-3)

    fig.canvas.draw()

for w in [sliderT0, sliderT, sliderWd, sliderDJ, sliderDR, sliderD0, sliderPoleK, radioU]:
    w.observe(calcSystem, names='value')

calcSystem(None)

bFeedForward = VBox([Label(value='Steuerung'), sliderWd, sliderT0, sliderT])
bFeedForward.layout = make_box_layout()
bInput = VBox([Label(value='Eingang'), radioU, sliderPoleK])
bInput.layout = make_box_layout()
bModel = VBox([Label(value='Strecke'), sliderDJ, sliderDR, sliderD0])
bModel.layout = make_box_layout()
controls = HBox([bFeedForward, bInput, bModel])

VBox([controls, output], layout=Layout(display='flex', flex_flow='column', justify_content='center', align_items='center'))